# Before Running This Notebook:

Go through the scraping process to gather your dataset of images containing subjects that you want to train into a YOLO model.  Then in groups, label these images using the local conda 'Label Studio' process outlined in Notion.

After you have successfully exported the labelled dataset from 'Label Studio', then continue to the steps below to begin the training process.

# Step 1: Prepare Training Data in your Google Drive
- Create a 'Root Directory Folder' in your Google Drive
- Create this file structure in your Google Drive
  - `/content/drive/MyDrive/YOLO/scrape/yolo_training_extinguisher/images`
  - `/content/drive/MyDrive/YOLO/scrape/yolo_training_extinguisher/labels`
  - `/content/drive/MyDrive/YOLO/scrape/data`


# Step 2: Scrape, Browse, Generate, Photograph, or find images
- You need to source at least 150-200 images
- They can be any size or resolution
- Minimize duplicates and overly complex images
- Prioritize images in the context you expect to track from using your trained YOLO model.

In [ ]:
#@markdown ## Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#@markdown ## Check you are running a CUDA compatible GPU
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import os
import shutil
from pathlib import Path
import random
from typing import List, Dict, Tuple

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def _norm_stem(p: Path) -> str:
    return p.stem.strip().lower()

def _gather_images_in_dir(images_dir: Path, recursive: bool) -> Dict[str, Path]:
    it = images_dir.rglob("*") if recursive else images_dir.iterdir()
    img_map = {}
    for p in it:
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            img_map.setdefault(_norm_stem(p), p)
    return img_map

def _gather_labels(labels_dir: Path, recursive: bool) -> Dict[str, Path]:
    it = labels_dir.rglob("*") if recursive else labels_dir.iterdir()
    lab_map = {}
    for p in it:
        if p.is_file() and p.suffix.lower() == ".txt":
            lab_map[_norm_stem(p)] = p
    return lab_map

def _place(src_path: Path, dst_path: Path, mode: str):
    if mode == "copy":
        shutil.copy2(src_path, dst_path)
    elif mode == "hardlink":
        os.link(src_path, dst_path)
    elif mode == "symlink":
        dst_path.symlink_to(src_path)
    else:
        raise ValueError("copy_mode must be 'copy', 'hardlink', or 'symlink'.")

def split_yolo_robust(
    images_dir: str,
    labels_dir: str,
    dest_root: str,
    train_ratio: float = 0.9,
    seed: int = 42,
    val_folder_name: str = "validation",
    overwrite: bool = False,
    copy_mode: str = "copy",
    dry_run: bool = False,
    recursive_sources: bool = True,
):
    images_dir = Path(images_dir).resolve()
    labels_dir = Path(labels_dir).resolve()
    dst = Path(dest_root).resolve()

    if not labels_dir.is_dir():
        raise FileNotFoundError(f"Missing labels folder: {labels_dir}")
    if not images_dir.is_dir():
        raise FileNotFoundError(f"Missing images folder: {images_dir}")

    for s in (images_dir, labels_dir):
        if str(dst).startswith(str(s)):
            raise ValueError("Destination must NOT be inside source folders.")
        if str(s).startswith(str(dst)):
            raise ValueError("Source folders must NOT be inside destination.")

    if dst.exists():
        if overwrite:
            if dry_run:
                print(f"[DRY RUN] Would remove {dst}")
            else:
                shutil.rmtree(dst)
        else:
            raise FileExistsError(f"Destination '{dst}' exists. Set overwrite=True to replace it.")

    lab_map = _gather_labels(labels_dir, recursive_sources)
    if not lab_map:
        raise RuntimeError("No label files (.txt) found in labels_dir.")
    img_map = _gather_images_in_dir(images_dir, recursive_sources)

    inter = set(img_map.keys()).intersection(lab_map.keys())
    if not inter:
        # Diagnostics
        imgs_count = len(img_map)
        labs_count = len(lab_map)
        print("No valid pairs. Diagnostics:")
        print("  images found:", imgs_count)
        print("  labels found:", labs_count)
        print("  sample image files:", [img_map[k].name for k in list(img_map.keys())[:10]] if imgs_count else [])
        print("  sample label files:", [lab_map[k].name for k in list(lab_map.keys())[:10]])
        only_imgs = sorted(set(img_map) - set(lab_map))
        only_labs = sorted(set(lab_map) - set(img_map))
        print("  unmatched image stems (first 20):", [k for k in only_imgs[:20]])
        print("  unmatched label stems (first 20):", [lab_map[k].name for k in only_labs[:20]])
        raise RuntimeError("No valid image/label pairs found in the supplied directories. "
                           "Ensure stems match and images actually exist in images_dir.")

    pairs: List[Tuple[Path, Path]] = [(img_map[k], lab_map[k]) for k in sorted(inter)]

    random.seed(seed)
    random.shuffle(pairs)
    n = len(pairs)
    n_train = int(round(n * train_ratio))
    if n > 1:
        n_train = min(max(n_train, 1), n - 1)
    train_pairs, val_pairs = pairs[:n_train], pairs[n_train:]

    to_make = [
        dst / "train" / "images",
        dst / "train" / "labels",
        dst / val_folder_name / "images",
        dst / val_folder_name / "labels",
    ]
    for d in to_make:
        if dry_run:
            print(f"[DRY RUN] Would create: {d}")
        else:
            d.mkdir(parents=True, exist_ok=True)

    def _write_pairs(pairs_list, split_name):
        img_out = dst / split_name / "images"
        lbl_out = dst / split_name / "labels"
        for img_path, lbl_path in pairs_list:
            if dry_run:
                print(f"[DRY RUN] Would {copy_mode}: {img_path} -> {img_out / img_path.name}")
                print(f"[DRY RUN] Would {copy_mode}: {lbl_path} -> {lbl_out / lbl_path.name}")
            else:
                _place(img_path, img_out / img_path.name, copy_mode)
                _place(lbl_path, lbl_out / lbl_path.name, copy_mode)

    _write_pairs(train_pairs, "train")
    _write_pairs(val_pairs, val_folder_name)

    unmatched_imgs = sorted(set(img_map) - set(lab_map))
    unmatched_labs = sorted(set(lab_map) - set(img_map))

    print("==== YOLO Split Summary ====")
    print(f"Images source:     {images_dir}")
    print(f"Labels source:     {labels_dir}")
    print(f"Destination:       {dst}")
    print(f"Total matched:     {n}")
    print(f"Train pairs:       {len(train_pairs)} ({len(train_pairs)/n:.1%})")
    print(f"Val pairs:         {len(val_pairs)} ({len(val_pairs)/n:.1%})")
    if unmatched_imgs:
        print(f"\nImages without labels (ignored): {len(unmatched_imgs)} (showing up to 10)")
        for k in unmatched_imgs[:10]:
            print("  -", img_map[k].name)
    if unmatched_labs:
        print(f"\nLabels without images (ignored): {len(unmatched_labs)} (showing up to 10)")
        for k in unmatched_labs[:10]:
            print("  -", lab_map[k].name)
    print("\nDone.")

images_in = "/content/drive/MyDrive/YOLO/scrape/yolo_training_extinguisher/images"
labels_in = "/content/drive/MyDrive/YOLO/scrape/yolo_training_extinguisher/labels"
dest_out  = "/content/drive/MyDrive/YOLO/scrape/data"

split_yolo_robust(
    images_dir=images_in,
    labels_dir=labels_in,
    dest_root=dest_out,
    train_ratio=0.90,
    seed=42,
    val_folder_name="validation",
    overwrite=True,
    copy_mode="copy",
    dry_run=False,
    recursive_sources=True,
)


==== YOLO Split Summary (Non-destructive) ====
Images source:     /content/drive/MyDrive/YOLO/scrape/yolo_training_extinguisher/images
Labels source:     /content/drive/MyDrive/YOLO/scrape/yolo_training_extinguisher/labels
Destination:       /content/drive/MyDrive/YOLO/scrape/data
Total matched:     167
Train pairs:       150 (89.8%)
Val pairs:         17 (10.2%)

Done.


In [ ]:
#@markdown ## Install Ultralytics Dependencies
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.0 MB/s eta 0:00:00


In [ ]:
#@markdown ## Create the training `.yaml` file

import os
import json
import yaml
from pathlib import Path
from typing import Optional, Dict, Any

def create_ultralytics_data_yaml(
    dest_root: str,
    classes_txt_path: str,
    output_yaml_path: Optional[str] = None,
    val_folder_name: str = "validation",
    notes_json_path: Optional[str] = None,
):
    dest_root = Path(dest_root).resolve()
    classes_txt_path = Path(classes_txt_path).resolve()
    output_yaml_path = Path(output_yaml_path).resolve() if output_yaml_path else (dest_root / "data.yaml")

    train_images = dest_root / "train" / "images"
    val_images = dest_root / val_folder_name / "images"
    if not train_images.is_dir():
        raise FileNotFoundError(f"train/images not found: {train_images}")
    if not val_images.is_dir():
        raise FileNotFoundError(f"{val_folder_name}/images not found: {val_images}")
    if not classes_txt_path.is_file():
        raise FileNotFoundError(f"classes.txt not found at: {classes_txt_path}")

    classes = []
    with open(classes_txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                classes.append(line)
    if not classes:
        raise ValueError(f"No classes found inside {classes_txt_path}")

    meta: Dict[str, Any] = {}
    if notes_json_path:
        nj = Path(notes_json_path).resolve()
        if nj.exists() and nj.is_file():
            try:
                with open(nj, "r", encoding="utf-8") as jf:
                    raw = json.load(jf)
                def _sanitize(value, max_len=400):
                    if isinstance(value, (str, int, float, bool)) or value is None:
                        return value if not (isinstance(value, str) and len(value) > max_len) else value[:max_len] + "…"
                    if isinstance(value, list):
                        return [_sanitize(v) for v in value[:50]]
                    if isinstance(value, dict):
                        return {str(k): _sanitize(v) for k, v in list(value.items())[:50]}
                    return str(value)
                meta = _sanitize(raw)
            except Exception as e:
                print(f"[WARN] Could not read/parse notes.json: {e}")

    data = {
        "path": str(dest_root),
        "train": "train/images",
        "val": f"{val_folder_name}/images",
        "nc": len(classes),
        "names": classes,
    }
    if meta:
        data["meta"] = meta

    output_yaml_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

    print(f"data.yaml created at {output_yaml_path}")

dest_root = "/content/drive/MyDrive/YOLO/scrape/data"
classes_txt = "/content/drive/MyDrive/YOLO/scrape/yolo_training_extinguisher/classes.txt"
notes_json = "/content/drive/MyDrive/YOLO/scrape/yolo_training_extinguisher/notes.json"
out_yaml = None
val_folder_name = "validation"

create_ultralytics_data_yaml(
    dest_root=dest_root,
    classes_txt_path=classes_txt,
    output_yaml_path=out_yaml,
    val_folder_name=val_folder_name,
    notes_json_path=notes_json,
)


data.yaml created at /content/drive/MyDrive/YOLO/scrape/data/data.yaml


In [ ]:
# Colab training script with automatic CPU/GPU selection

# This will create a folder called '/content/drive/MyDrive/YOLO/scrape/runs'
# This folder will contain the epoches outputs which are the '.pts' file weights

!pip -q install ultralytics>=8.3.0

import json, yaml, torch
from pathlib import Path
from ultralytics import YOLO

# -------- Variables --------
BASE_DIR    = Path("/content/drive/MyDrive/YOLO/scrape")
DEST_ROOT   = BASE_DIR / "data"
DATA_YAML   = DEST_ROOT / "data.yaml"
MODEL_NAME  = "yolo11s.pt"
IMG_SIZE    = 640
PROJECT_DIR = BASE_DIR / "runs"
RUN_NAME    = "extinguisher_yolo11s"
# ---------------------------

HAS_CUDA = torch.cuda.is_available()
DEVICE = 0 if HAS_CUDA else "cpu"

if not HAS_CUDA:
    if IMG_SIZE > 512:
        IMG_SIZE = 512
    BATCH = 4
    WORKERS = 2
else:
    BATCH = -1
    WORKERS = 8

# Epoch heuristic (<=200 train images -> 60, else 40)
train_imgs_dir = DEST_ROOT / "train" / "images"
n_train_images = sum(1 for p in train_imgs_dir.rglob("*")
                     if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"})
EPOCHS = 60 if n_train_images <= 200 else 40

print(f"CUDA available: {HAS_CUDA}")
print(f"device: {DEVICE}")
print(f"data:   {DATA_YAML}")
print(f"model:  {MODEL_NAME}")
print(f"epochs: {EPOCHS} (train images: {n_train_images})")
print(f"imgsz:  {IMG_SIZE}")
print(f"batch:  {BATCH}")
print(f"workers:{WORKERS}")
print(f"runs:   {PROJECT_DIR}/{RUN_NAME}")

model = YOLO(MODEL_NAME)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    device=DEVICE,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    batch=BATCH,
    cache=True,
    patience=20,
    workers=WORKERS,
)


CUDA available: False
device: cpu
data:   /content/drive/MyDrive/YOLO/scrape/data/data.yaml
model:  yolo11s.pt
epochs: 60 (train images: 150)
imgsz:  512
batch:  4
workers:2
runs:   /content/drive/MyDrive/YOLO/scrape/runs/extinguisher_yolo11s
Ultralytics 8.3.221 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/YOLO/scrape/data/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78608fa094c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Colab training script with automatic CPU/GPU selection
# Saves a checkpoint EVERY epoch and prints the exact output folder + weights list.

!pip -q install "ultralytics>=8.3.0"

import json, yaml, torch, glob
from pathlib import Path
from ultralytics import YOLO

# -------- Variables --------
BASE_DIR    = Path("/content/drive/MyDrive/YOLO/scrape")
DEST_ROOT   = BASE_DIR / "data"
DATA_YAML   = DEST_ROOT / "data.yaml"
MODEL_NAME  = "yolo11s.pt"
IMG_SIZE    = 640
PROJECT_DIR = BASE_DIR / "runs"
RUN_NAME    = "extinguisher_yolo11s"
# ---------------------------

HAS_CUDA = torch.cuda.is_available()
DEVICE = 0 if HAS_CUDA else "cpu"

if not HAS_CUDA:
    if IMG_SIZE > 512:
        IMG_SIZE = 512
    BATCH = 4
    WORKERS = 2
else:
    BATCH = -1
    WORKERS = 8

# Epoch heuristic (<=200 train images -> 60 else 40)
train_imgs_dir = DEST_ROOT / "train" / "images"
valid_exts = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
n_train_images = sum(1 for p in train_imgs_dir.rglob("*")
                     if p.is_file() and p.suffix.lower() in valid_exts)
EPOCHS = 60 if n_train_images <= 200 else 40

print(f"CUDA available: {HAS_CUDA}")
print(f"device: {DEVICE}")
print(f"data:   {DATA_YAML}")
print(f"model:  {MODEL_NAME}")
print(f"epochs: {EPOCHS} (train images: {n_train_images})")
print(f"imgsz:  {IMG_SIZE}")
print(f"batch:  {BATCH}")
print(f"workers:{WORKERS}")
print(f"runs:   {PROJECT_DIR}/{RUN_NAME}")

model = YOLO(MODEL_NAME)
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    device=DEVICE,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    batch=BATCH,
    cache=True,
    patience=20,
    workers=WORKERS,
    save=True,
    save_period=5
)

# Print exact output path and list saved weights
try:
    save_dir = Path(model.trainer.save_dir)
except Exception:
    save_dir = PROJECT_DIR / "detect" / RUN_NAME

weights_dir = save_dir / "weights"
print("\nRun folder:     ", save_dir)
print("Weights folder: ", weights_dir)

weights = sorted(glob.glob(str(weights_dir / "*.pt")))
print("Saved weight files:")
for w in weights:
    print(" -", w)

